# E3 — Strong non-graph ranking baseline and interpretable disassembly heuristics
## Is a GNN actually needed, and how much better is learned ranking than simple rules?

E1 and E2 showed that cross-node interaction is useful, but the gain is not specific to the recorded physical fixation topology. E3 addresses the next reviewer question: whether the reported neural performance is meaningfully stronger than a competitive non-graph learner and simple manufacturing/topology heuristics.

### Conditions

All conditions are evaluated under the same strict 10-fold leave-one-product-out (LOPO) protocol.

1. **XGBRanker** — a gradient-boosted pairwise ranking model trained only on the same product-invariant operation attributes used by the neural models: current/candidate categorical operation descriptors, operation-local normalized degree features, equality/difference features, and normalized sequence progress. It receives no adjacency matrix, no component identity, and no product ID.
2. **Class-transition prior** — a training-only smoothed prior over which operation signature follows the current operation signature in the nine training batteries.
3. **Graph-distance heuristic** — ranks candidates by the minimum shortest-path distance between components touched by the current and candidate operations in the held-out battery's recorded fixation graph.
4. **Shared-component continuation heuristic** — prefers candidates touching one or more of the same components as the current operation.
5. **Uniform-random analytical baseline** — the exact expected ranking metrics for each candidate-set size.

### XGBRanker training

For each training state, all remaining candidate operations are included in the ranking group. The recorded immediate next operation has relevance label 1 and every later candidate has label 0. Training excludes the final singleton-only state, matching the neural pairwise-training logic.

The boosted ranker uses a fixed `rank:pairwise` objective and a fixed hyperparameter configuration across all folds; the held-out product is never used for tuning or model selection. Three fixed seeds `[17, 29, 43]` are used and averaged within each held-out product.

Categorical operation descriptors are one-hot encoded using fold-specific vocabularies learned only from the nine training batteries. Unseen held-out categories therefore map to the same reserved `<UNK>` indices used by the neural experiments.

### Metrics

The primary endpoint remains Hits@1, with Hits@3 and MRR as supporting metrics. Ties use the same expected-value rule as the main manuscript.

E3 additionally reports:

- product-level skill score: `(Hits@1 - random Hits@1) / (1 - random Hits@1)`;
- descriptive performance stratified by candidate-set size;
- exact product-level paired comparisons among the learned ranker and the three heuristics.

The held-out battery remains the independent statistical unit (`n=10`). This is a post-primary robustness/baseline analysis.


In [ ]:
# 1. Imports and frozen configuration

from pathlib import Path
from collections import Counter, defaultdict
import tarfile, json, math, random, time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

SEEDS = [17, 29, 43]
SHUFFLE_IDS = [1, 2, 3, 4, 5]

EPOCHS = 30
NEGATIVES_PER_STATE = 8
LR = 2e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.10

EMBED_DIM = 12
HIDDEN_DIM = 64
GNN_DIM = 32

BASE_SEED = 20261003
GRAPH_SHUFFLE_BASE = 20261119

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_FITS = 10 * len(SEEDS)

print("Device:", DEVICE)
print("Torch:", torch.__version__)
print("Planned XGBRanker fits:", N_FITS)

if DEVICE.type != "cuda":
    print("WARNING: GPU is strongly recommended for this notebook.")

In [ ]:
# 2. Find the KIT dataset automatically.
#
# The cell first searches for already-extracted CSVs anywhere under /content.
# If not found, it searches for the uploaded KIT .tar archive and extracts it.

root = Path("/content")

catalogues = list(root.rglob("battery_catalogue.csv"))

if catalogues:
    DATA = catalogues[0].parent
else:
    archives = list(root.rglob("10.35097-emz24pksshndq468.tar"))
    if not archives:
        archives = list(root.rglob("10.35097-*.tar"))

    assert archives, (
        "KIT dataset not found. Upload 10.35097-emz24pksshndq468.tar "
        "to the Colab Files panel, then rerun this cell."
    )

    TAR_PATH = archives[0]
    print("Using archive:", TAR_PATH)

    work = root / "battery_disassembly_dataset_extracted"
    work.mkdir(parents=True, exist_ok=True)

    with tarfile.open(TAR_PATH, "r") as tf:
        tf.extractall(work)

    catalogues = list(work.rglob("battery_catalogue.csv"))
    assert catalogues, "battery_catalogue.csv not found after extraction."
    DATA = catalogues[0].parent

print("Dataset directory:", DATA)

In [ ]:
# 3. Load and validate raw data.

bat = pd.read_csv(DATA / "battery_catalogue.csv")
parts = pd.read_csv(DATA / "parts.csv")
fix = pd.read_csv(DATA / "fixations.csv")
ops = pd.read_csv(DATA / "operations.csv")

print("Batteries:", len(bat))
print("Parts:", len(parts))
print("Fixations:", len(fix))
print("Operations:", len(ops))

display(bat[["id", "name"]])

In [ ]:
# 4. Attach each operation to a battery and reconstruct the 10 reference chains.

battery_names = bat.set_index("id")["name"].to_dict()

fix_battery = fix.set_index("id")["battery_id"].to_dict()
part_battery = parts.set_index("id")["battery_id"].to_dict()
part_class_raw = parts.set_index("id")["class"].to_dict()

fix_info_raw = (
    fix.set_index("id")[["class", "fromPartID", "toPartID"]]
    .to_dict("index")
)

def operation_battery(row):
    if pd.notna(row.fixation_id):
        return int(fix_battery[int(row.fixation_id)])
    return int(part_battery[int(row.part_id)])

ops["battery_id"] = ops.apply(operation_battery, axis=1)
ops_by_id = ops.set_index("id")

chains = {}

for b in sorted(ops["battery_id"].unique()):
    sub = ops[ops["battery_id"] == b]
    ids = set(sub["id"].astype(int))
    next_ids = set(sub["next_op_id"].dropna().astype(int))

    starts = list(ids - next_ids)
    terminals = sub[sub["next_op_id"].isna()]["id"].astype(int).tolist()

    assert len(starts) == 1
    assert len(terminals) == 1

    seq = []
    seen = set()
    cur = int(starts[0])

    while cur in ids and cur not in seen:
        seen.add(cur)
        seq.append(cur)

        nxt = ops_by_id.loc[cur, "next_op_id"]
        if pd.isna(nxt):
            break
        cur = int(nxt)

    assert len(seq) == len(ids), f"Broken chain for battery {b}"
    chains[int(b)] = seq

print({
    battery_names[b]: len(seq)
    for b, seq in chains.items()
})

In [ ]:
# 5. Raw full relational signatures.
#
# These are used only for diagnostics (seen vs unseen motif/signature).
# They are NOT output classes for the neural ranking models.

def raw_full_signature(op_id):
    r = ops_by_id.loc[int(op_id)]
    skill = str(r.skill_id)

    if pd.notna(r.fixation_id):
        fi = fix_info_raw[int(r.fixation_id)]
        src_cls = str(part_class_raw[int(fi["fromPartID"])])
        dst_cls = str(part_class_raw[int(fi["toPartID"])])
        return (
            f"{skill}|fix:{fi['class']}|"
            f"{src_cls}->{dst_cls}"
        )

    return (
        f"{skill}|part:"
        f"{part_class_raw[int(r.part_id)]}"
    )

raw_signatures = {
    b: [raw_full_signature(op_id) for op_id in seq]
    for b, seq in chains.items()
}

In [ ]:
# 6. Fold-specific vocabularies.
#
# Vocabularies are learned ONLY from the 9 training batteries.
# Category 0 is <UNK>, so truly unseen test categories do not leak into training.

def make_vocab(values):
    vals = sorted({
        str(x)
        for x in values
        if pd.notna(x)
    })
    return {
        "<UNK>": 0,
        **{v: i + 1 for i, v in enumerate(vals)}
    }

def lookup(vocab, value):
    if pd.isna(value):
        return 0
    return vocab.get(str(value), 0)

def build_fold_vocab(train_batteries):
    return {
        "part": make_vocab(
            parts[
                parts["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "fix": make_vocab(
            fix[
                fix["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "skill": make_vocab(
            ops[
                ops["battery_id"].isin(train_batteries)
            ]["skill_id"]
        ),
    }

In [ ]:
# 7. Convert one battery into product-invariant graph + operation features.
#
# graph_mode:
#   "real"    -> true physical fixation topology
#   "shuffle" -> fixed permutation of topology relative to semantic/operation nodes
#   "noedge"  -> same GNN architecture, but no message-passing edges
#
# Operation-local attributes are identical across all graph modes.
# Therefore only the graph branch is manipulated.

def build_battery_pack(
    battery_id,
    vocab,
    graph_mode="real",
    shuffle_id=0,
):
    assert graph_mode in {"real", "shuffle", "noedge"}

    psub = parts[
        parts["battery_id"] == battery_id
    ].copy()

    fsub = fix[
        fix["battery_id"] == battery_id
    ].copy()

    part_id_to_local = {
        int(pid): i
        for i, pid in enumerate(psub["id"].astype(int))
    }

    n_nodes = len(psub)

    true_degree = np.zeros(n_nodes, dtype=np.float32)

    edge_src = []
    edge_dst = []
    edge_type = []

    for _, r in fsub.iterrows():
        a = part_id_to_local[int(r.fromPartID)]
        b = part_id_to_local[int(r.toPartID)]

        true_degree[a] += 1
        true_degree[b] += 1

        # Undirected message-passing representation.
        edge_src += [a, b]
        edge_dst += [b, a]

        et = lookup(vocab["fix"], r["class"])
        edge_type += [et, et]

    true_degree_norm = (
        true_degree / max(1.0, float(true_degree.max()))
    )

    node_part_class = np.array(
        [
            lookup(vocab["part"], cls)
            for cls in psub["class"]
        ],
        dtype=np.int64,
    )

    true_edge_index = np.array(
        [edge_src, edge_dst],
        dtype=np.int64,
    )

    true_edge_type = np.array(
        edge_type,
        dtype=np.int64,
    )

    if graph_mode == "real":
        graph_edge_index = true_edge_index.copy()
        graph_edge_type = true_edge_type.copy()
        graph_degree_norm = true_degree_norm.copy()

    elif graph_mode == "shuffle":
        # A deterministic, independent permutation for each
        # (shuffle_id, battery). This preserves the unlabeled graph,
        # but reassigns its physical locations to semantic nodes.
        rng_graph = np.random.default_rng(
            GRAPH_SHUFFLE_BASE
            + 10007 * int(shuffle_id)
            + int(battery_id)
        )

        permutation = rng_graph.permutation(n_nodes)

        graph_edge_index = permutation[true_edge_index]
        graph_edge_type = true_edge_type.copy()

        shuffled_degree = np.bincount(
            graph_edge_index.reshape(-1),
            minlength=n_nodes,
        ).astype(np.float32)

        graph_degree_norm = (
            shuffled_degree
            / max(1.0, float(shuffled_degree.max()))
        )

    else:  # noedge
        graph_edge_index = np.empty((2, 0), dtype=np.int64)
        graph_edge_type = np.empty((0,), dtype=np.int64)

        # Do not leak adjacency through the graph branch.
        graph_degree_norm = np.zeros(
            n_nodes,
            dtype=np.float32,
        )

    seq = [int(x) for x in chains[battery_id]]

    categorical = []
    graph_nodes = []
    numeric = []
    signatures = []

    for op_id in seq:
        r = ops_by_id.loc[op_id]

        skill_idx = lookup(
            vocab["skill"],
            r.skill_id,
        )

        if pd.notna(r.fixation_id):
            fi = fix_info_raw[int(r.fixation_id)]

            src_pid = int(fi["fromPartID"])
            dst_pid = int(fi["toPartID"])

            src_node = part_id_to_local[src_pid]
            dst_node = part_id_to_local[dst_pid]

            categorical.append([
                skill_idx,
                0,
                lookup(vocab["fix"], fi["class"]),
                lookup(vocab["part"], part_class_raw[src_pid]),
                lookup(vocab["part"], part_class_raw[dst_pid]),
                0,
            ])

            graph_nodes.append([
                src_node,
                dst_node,
                -1,
            ])

            # IMPORTANT:
            # identical operation-local features in every graph condition.
            numeric.append([
                true_degree_norm[src_node],
                true_degree_norm[dst_node],
                0.0,
            ])

        else:
            pid = int(r.part_id)
            node = part_id_to_local[pid]

            categorical.append([
                skill_idx,
                1,
                0,
                0,
                0,
                lookup(vocab["part"], part_class_raw[pid]),
            ])

            graph_nodes.append([
                -1,
                -1,
                node,
            ])

            numeric.append([
                0.0,
                0.0,
                true_degree_norm[node],
            ])

        signatures.append(raw_full_signature(op_id))

    return {
        "battery_id": int(battery_id),
        "battery_name": battery_names[int(battery_id)],
        "node_part_class": node_part_class,
        "node_degree": graph_degree_norm.astype(np.float32),
        "edge_index": graph_edge_index,
        "edge_type": graph_edge_type,
        "op_categorical": np.asarray(categorical, dtype=np.int64),
        "op_graph_nodes": np.asarray(graph_nodes, dtype=np.int64),
        "op_numeric": np.asarray(numeric, dtype=np.float32),
        "full_signatures": signatures,
        "n_ops": len(seq),
        "graph_mode": graph_mode,
        "shuffle_id": int(shuffle_id),
    }

In [ ]:
# 8. Neural models.

class BaseOperationRanker(nn.Module):
    """
    Product-invariant MLP operation ranker.

    It never receives battery/product IDs or raw entity IDs.
    """

    def __init__(
        self,
        n_skill,
        n_fix,
        n_part,
        emb_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.skill_emb = nn.Embedding(n_skill, emb_dim)
        self.kind_emb = nn.Embedding(2, 4)
        self.fix_emb = nn.Embedding(n_fix, emb_dim)
        self.part_emb = nn.Embedding(n_part, emb_dim)

        op_input_dim = (
            emb_dim      # skill
            + 4          # operation kind
            + emb_dim    # fixation class
            + emb_dim    # source part class
            + emb_dim    # target part class
            + emb_dim    # removed part class
            + 3          # normalized local degrees
        )

        self.op_mlp = nn.Sequential(
            nn.Linear(op_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # current, candidate, abs difference, product, progress
        self.score_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 4 + 1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def base_operation_encoding(self, pack):
        cats = torch.as_tensor(
            pack["op_categorical"],
            dtype=torch.long,
            device=DEVICE,
        )

        nums = torch.as_tensor(
            pack["op_numeric"],
            dtype=torch.float32,
            device=DEVICE,
        )

        skill = cats[:, 0]
        kind = cats[:, 1]
        fix_class = cats[:, 2]
        src_class = cats[:, 3]
        dst_class = cats[:, 4]
        part_class = cats[:, 5]

        x = torch.cat([
            self.skill_emb(skill),
            self.kind_emb(kind),
            self.fix_emb(fix_class),
            self.part_emb(src_class),
            self.part_emb(dst_class),
            self.part_emb(part_class),
            nums,
        ], dim=-1)

        return self.op_mlp(x)

    def encode_operations(self, pack):
        return self.base_operation_encoding(pack)

    def score_pairs(self, current_repr, candidate_repr, progress):
        x = torch.cat([
            current_repr,
            candidate_repr,
            torch.abs(current_repr - candidate_repr),
            current_repr * candidate_repr,
            progress,
        ], dim=-1)

        return self.score_mlp(x).squeeze(-1)


class GraphEncoder(nn.Module):
    """
    Two-layer product-invariant message-passing network.
    """

    def __init__(
        self,
        n_part,
        n_fix,
        part_emb_dim=EMBED_DIM,
        edge_emb_dim=8,
        graph_dim=GNN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.part_emb = nn.Embedding(n_part, part_emb_dim)
        self.edge_emb = nn.Embedding(n_fix, edge_emb_dim)

        self.input_proj = nn.Linear(
            part_emb_dim + 1,
            graph_dim,
        )

        self.message_layers = nn.ModuleList([
            nn.Linear(
                graph_dim + edge_emb_dim,
                graph_dim,
            )
            for _ in range(2)
        ])

        self.self_layers = nn.ModuleList([
            nn.Linear(graph_dim, graph_dim)
            for _ in range(2)
        ])

        self.dropout = dropout

    def forward(self, pack):
        node_class = torch.as_tensor(
            pack["node_part_class"],
            dtype=torch.long,
            device=DEVICE,
        )

        degree = torch.as_tensor(
            pack["node_degree"],
            dtype=torch.float32,
            device=DEVICE,
        ).unsqueeze(-1)

        h = F.relu(
            self.input_proj(
                torch.cat([
                    self.part_emb(node_class),
                    degree,
                ], dim=-1)
            )
        )

        edge_index = torch.as_tensor(
            pack["edge_index"],
            dtype=torch.long,
            device=DEVICE,
        )

        edge_type = torch.as_tensor(
            pack["edge_type"],
            dtype=torch.long,
            device=DEVICE,
        )

        src = edge_index[0]
        dst = edge_index[1]

        edge_embedding = self.edge_emb(edge_type)

        for message_layer, self_layer in zip(
            self.message_layers,
            self.self_layers,
        ):
            msg = message_layer(
                torch.cat([
                    h[src],
                    edge_embedding,
                ], dim=-1)
            )

            agg = torch.zeros(
                (h.shape[0], h.shape[1]),
                dtype=h.dtype,
                device=DEVICE,
            )

            agg.index_add_(0, dst, msg)

            counts = torch.zeros(
                (h.shape[0], 1),
                dtype=h.dtype,
                device=DEVICE,
            )

            counts.index_add_(
                0,
                dst,
                torch.ones(
                    (dst.numel(), 1),
                    dtype=h.dtype,
                    device=DEVICE,
                ),
            )

            agg = agg / counts.clamp_min(1.0)

            h = F.relu(
                self_layer(h) + agg
            )

            h = F.dropout(
                h,
                p=self.dropout,
                training=self.training,
            )

        return h


class GNNOperationRanker(BaseOperationRanker):

    def __init__(
        self,
        n_skill,
        n_fix,
        n_part,
        emb_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        graph_dim=GNN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__(
            n_skill=n_skill,
            n_fix=n_fix,
            n_part=n_part,
            emb_dim=emb_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

        self.graph_encoder = GraphEncoder(
            n_part=n_part,
            n_fix=n_fix,
            part_emb_dim=emb_dim,
            graph_dim=graph_dim,
            dropout=dropout,
        )

        # src embedding, dst embedding, absolute difference
        self.structural_projection = nn.Sequential(
            nn.Linear(graph_dim * 3, hidden_dim),
            nn.ReLU(),
        )

        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
        )

    def encode_operations(self, pack):
        base = self.base_operation_encoding(pack)
        node_h = self.graph_encoder(pack)

        op_nodes = torch.as_tensor(
            pack["op_graph_nodes"],
            dtype=torch.long,
            device=DEVICE,
        )

        n_ops = op_nodes.shape[0]

        a = torch.zeros(
            (n_ops, node_h.shape[1]),
            device=DEVICE,
        )

        b = torch.zeros_like(a)

        fixation_mask = op_nodes[:, 0] >= 0

        if fixation_mask.any():
            a[fixation_mask] = node_h[
                op_nodes[fixation_mask, 0]
            ]

            b[fixation_mask] = node_h[
                op_nodes[fixation_mask, 1]
            ]

        part_mask = op_nodes[:, 2] >= 0

        if part_mask.any():
            a[part_mask] = node_h[
                op_nodes[part_mask, 2]
            ]

        structural = self.structural_projection(
            torch.cat([
                a,
                b,
                torch.abs(a - b),
            ], dim=-1)
        )

        return self.combine(
            torch.cat([
                base,
                structural,
            ], dim=-1)
        )

In [ ]:
# 9. Reproducibility + pairwise-ranking training

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_training_pairs(n_ops, k, rng):
    current, positive, negative, progress = [], [], [], []

    for t in range(n_ops - 2):
        pool = np.arange(
            t + 2,
            n_ops,
            dtype=np.int64,
        )

        neg = rng.choice(
            pool,
            size=k,
            replace=(len(pool) < k),
        )

        current.append(t)
        positive.append(t + 1)
        negative.append(neg)
        progress.append(
            t / max(1, n_ops - 1)
        )

    return (
        np.asarray(current, dtype=np.int64),
        np.asarray(positive, dtype=np.int64),
        np.stack(negative),
        np.asarray(progress, dtype=np.float32),
    )


def build_gnn(vocab):
    model = GNNOperationRanker(
        n_skill=len(vocab["skill"]),
        n_fix=len(vocab["fix"]),
        n_part=len(vocab["part"]),
    )
    return model.to(DEVICE)


def train_one_gnn_condition(
    train_batteries,
    test_battery,
    seed,
    graph_mode,
    shuffle_id=0,
):
    set_seed(seed)

    vocab = build_fold_vocab(train_batteries)

    packs = {
        b: build_battery_pack(
            b,
            vocab,
            graph_mode=graph_mode,
            shuffle_id=shuffle_id,
        )
        for b in train_batteries + [test_battery]
    }

    model = build_gnn(vocab)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    # Using the same training seed across paired graph conditions
    # keeps initialization, training order, and negative sampling paired.
    rng = np.random.default_rng(seed)

    for epoch in range(EPOCHS):
        order = list(train_batteries)
        rng.shuffle(order)

        model.train()

        for b in order:
            pack = packs[b]

            cur, pos, neg, prog = make_training_pairs(
                pack["n_ops"],
                NEGATIVES_PER_STATE,
                rng,
            )

            z = model.encode_operations(pack)

            cur_t = torch.as_tensor(
                cur,
                dtype=torch.long,
                device=DEVICE,
            )

            pos_t = torch.as_tensor(
                pos,
                dtype=torch.long,
                device=DEVICE,
            )

            neg_t = torch.as_tensor(
                neg,
                dtype=torch.long,
                device=DEVICE,
            )

            prog_t = torch.as_tensor(
                prog,
                dtype=torch.float32,
                device=DEVICE,
            ).unsqueeze(-1)

            cur_z = z[cur_t]
            pos_z = z[pos_t]

            pos_score = model.score_pairs(
                cur_z,
                pos_z,
                prog_t,
            )

            k = neg_t.shape[1]

            cur_rep = (
                cur_z[:, None, :]
                .expand(-1, k, -1)
                .reshape(-1, cur_z.shape[1])
            )

            neg_z = z[neg_t.reshape(-1)]

            prog_rep = (
                prog_t[:, None, :]
                .expand(-1, k, -1)
                .reshape(-1, 1)
            )

            neg_score = model.score_pairs(
                cur_rep,
                neg_z,
                prog_rep,
            ).view(len(cur), k)

            loss = F.softplus(
                -(pos_score[:, None] - neg_score)
            ).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return model, packs[test_battery], vocab

In [ ]:
# 10. Tie-aware ranking metrics.

def expected_tie_metrics(scores, target_position=0, tol=1e-8):
    """
    Scores correspond to candidate operations.

    Returns the expected Hits@1, Hits@3, and reciprocal rank
    under random ordering within exact/near score ties.
    """

    scores = np.asarray(scores)
    target_score = scores[target_position]

    greater = int(
        np.sum(scores > target_score + tol)
    )

    equal = int(
        np.sum(
            np.abs(scores - target_score) <= tol
        )
    )

    equal = max(equal, 1)

    hits1 = (
        max(
            0,
            min(equal, 1 - greater),
        )
        / equal
        if greater < 1
        else 0.0
    )

    hits3 = (
        max(
            0,
            min(equal, 3 - greater),
        )
        / equal
        if greater < 3
        else 0.0
    )

    ranks = np.arange(
        greater + 1,
        greater + equal + 1,
        dtype=float,
    )

    mrr = float(
        np.mean(1.0 / ranks)
    )

    return hits1, hits3, mrr

In [ ]:
# 11. Evaluate one neural model on an unseen battery.

def training_signature_set(train_batteries):
    s = set()

    for b in train_batteries:
        s.update(raw_signatures[b])

    return s


def evaluate_neural_model(
    model,
    test_pack,
    train_batteries,
):
    model.eval()

    seen_signatures = training_signature_set(
        train_batteries
    )

    rows = []

    with torch.no_grad():
        operation_repr = model.encode_operations(
            test_pack
        )

        n_ops = test_pack["n_ops"]

        for t in range(n_ops - 1):
            candidate_indices = np.arange(
                t + 1,
                n_ops,
                dtype=np.int64,
            )

            candidate_t = torch.as_tensor(
                candidate_indices,
                dtype=torch.long,
                device=DEVICE,
            )

            current_repr = (
                operation_repr[t]
                .unsqueeze(0)
                .expand(
                    len(candidate_indices),
                    -1,
                )
            )

            candidate_repr = operation_repr[
                candidate_t
            ]

            progress = torch.full(
                (len(candidate_indices), 1),
                t / max(1, n_ops - 1),
                dtype=torch.float32,
                device=DEVICE,
            )

            scores = model.score_pairs(
                current_repr,
                candidate_repr,
                progress,
            ).cpu().numpy()

            # candidate_indices[0] is the reference next operation.
            h1, h3, mrr = expected_tie_metrics(
                scores,
                target_position=0,
            )

            target_signature = (
                test_pack["full_signatures"][t + 1]
            )

            rows.append({
                "position": t,
                "hits1": h1,
                "hits3": h3,
                "mrr": mrr,
                "signature_seen_in_training":
                    target_signature in seen_signatures,
            })

    detail = pd.DataFrame(rows)

    metrics = {
        "hits1": float(detail["hits1"].mean()),
        "hits3": float(detail["hits3"].mean()),
        "mrr": float(detail["mrr"].mean()),
        "seen_signature_rate": float(
            detail["signature_seen_in_training"].mean()
        ),
    }

    seen = detail[
        detail["signature_seen_in_training"]
    ]

    unseen = detail[
        ~detail["signature_seen_in_training"]
    ]

    for label, subset in [
        ("seen", seen),
        ("unseen", unseen),
    ]:
        if len(subset) > 0:
            metrics[f"{label}_hits1"] = float(
                subset["hits1"].mean()
            )

            metrics[f"{label}_hits3"] = float(
                subset["hits3"].mean()
            )

            metrics[f"{label}_mrr"] = float(
                subset["mrr"].mean()
            )
        else:
            metrics[f"{label}_hits1"] = np.nan
            metrics[f"{label}_hits3"] = np.nan
            metrics[f"{label}_mrr"] = np.nan

    return metrics, detail

In [ ]:
# 12. Exact-capacity validity helper

def count_trainable_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

## 13. Imports and fixed E3 configuration

The boosted ranker deliberately uses a fixed configuration rather than selecting hyperparameters on the held-out product. Later, if E3 shows that tree-model tuning is consequential, a nested training-only tuning analysis can be added; no such tuning is needed to interpret the simple heuristics.


In [ ]:
# 13. E3 dependencies and frozen boosted-ranker configuration

import sys
import subprocess
import itertools
import shutil
from collections import Counter, defaultdict, deque

try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "xgboost"]
    )
    import xgboost as xgb

from sklearn.preprocessing import OneHotEncoder
from scipy import sparse
from packaging.version import Version

XGB_PARAMS = {
    "objective": "rank:pairwise",
    "eval_metric": "ndcg@3",
    "n_estimators": 250,
    "max_depth": 6,
    "learning_rate": 0.05,
    "min_child_weight": 1.0,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "tree_method": "hist",
    "n_jobs": -1,
}

if torch.cuda.is_available():
    if Version(xgb.__version__) >= Version("2.0.0"):
        XGB_PARAMS["device"] = "cuda"
    else:
        XGB_PARAMS["tree_method"] = "gpu_hist"

CANDIDATE_BINS = [
    (1, 5, "1-5"),
    (6, 10, "6-10"),
    (11, 25, "11-25"),
    (26, 50, "26-50"),
    (51, 100, "51-100"),
    (101, 10**9, ">100"),
]

print("XGBoost:", xgb.__version__)
print("XGB parameters:", XGB_PARAMS)


## 14. Pair-feature construction for XGBRanker

Each candidate is represented relative to the current operation using:

- six categorical descriptors for the current operation;
- six categorical descriptors for the candidate;
- three current local-degree features;
- three candidate local-degree features;
- absolute differences of the three degree features;
- six categorical equality indicators;
- normalized sequence progress.

No adjacency, node identity, battery identity, raw operation ID, or future position is supplied.


In [ ]:
# 14. Fold-specific one-hot encoder and pair-feature matrix construction

def make_pair_encoder(vocab):
    n_skill = len(vocab["skill"])
    n_fix = len(vocab["fix"])
    n_part = len(vocab["part"])

    single_op_categories = [
        np.arange(n_skill),
        np.arange(2),
        np.arange(n_fix),
        np.arange(n_part),
        np.arange(n_part),
        np.arange(n_part),
    ]

    categories = single_op_categories + [
        x.copy() for x in single_op_categories
    ]

    kwargs = {
        "categories": categories,
        "handle_unknown": "ignore",
        "dtype": np.float32,
    }

    try:
        encoder = OneHotEncoder(
            sparse_output=True,
            **kwargs,
        )
    except TypeError:
        encoder = OneHotEncoder(
            sparse=True,
            **kwargs,
        )

    encoder.fit(
        np.zeros((1, 12), dtype=np.int64)
    )
    return encoder


def pair_feature_matrix(pack, t, candidate_indices, encoder):
    candidate_indices = np.asarray(
        candidate_indices,
        dtype=np.int64,
    )

    n = len(candidate_indices)

    cur_cat = np.repeat(
        pack["op_categorical"][t:t+1],
        n,
        axis=0,
    )
    cand_cat = pack["op_categorical"][candidate_indices]

    pair_cat = np.concatenate(
        [cur_cat, cand_cat],
        axis=1,
    )

    cat_sparse = encoder.transform(pair_cat)

    cur_num = np.repeat(
        pack["op_numeric"][t:t+1],
        n,
        axis=0,
    ).astype(np.float32)

    cand_num = (
        pack["op_numeric"][candidate_indices]
        .astype(np.float32)
    )

    abs_num_diff = np.abs(
        cur_num - cand_num
    ).astype(np.float32)

    same_cat = (
        cur_cat == cand_cat
    ).astype(np.float32)

    progress = np.full(
        (n, 1),
        t / max(1, pack["n_ops"] - 1),
        dtype=np.float32,
    )

    numeric = np.concatenate(
        [
            cur_num,
            cand_num,
            abs_num_diff,
            same_cat,
            progress,
        ],
        axis=1,
    ).astype(np.float32)

    return sparse.hstack(
        [
            cat_sparse,
            sparse.csr_matrix(numeric),
        ],
        format="csr",
        dtype=np.float32,
    )


def build_xgb_training_data(
    train_batteries,
    vocab,
):
    encoder = make_pair_encoder(vocab)

    X_parts = []
    y_parts = []
    qid_parts = []

    query_id = 0

    for b in train_batteries:
        pack = build_battery_pack(
            b,
            vocab,
            graph_mode="real",
            shuffle_id=0,
        )

        n_ops = pack["n_ops"]

        # Exclude the final singleton-only state.
        for t in range(n_ops - 2):
            candidate_indices = np.arange(
                t + 1,
                n_ops,
                dtype=np.int64,
            )

            Xq = pair_feature_matrix(
                pack,
                t,
                candidate_indices,
                encoder,
            )

            yq = np.zeros(
                len(candidate_indices),
                dtype=np.float32,
            )
            yq[0] = 1.0

            X_parts.append(Xq)
            y_parts.append(yq)
            qid_parts.append(
                np.full(
                    len(candidate_indices),
                    query_id,
                    dtype=np.int32,
                )
            )

            query_id += 1

    X = sparse.vstack(
        X_parts,
        format="csr",
        dtype=np.float32,
    )
    y = np.concatenate(y_parts)
    qid = np.concatenate(qid_parts)

    return X, y, qid, encoder


def build_xgb_model(seed):
    params = dict(XGB_PARAMS)
    params["random_state"] = int(seed)

    return xgb.XGBRanker(
        **params,
    )


## 15. XGBRanker evaluation

Evaluation uses every candidate at every held-out decision state and exactly the same tie-aware Hits@1, Hits@3, and MRR definitions as the neural experiments.


In [ ]:
# 15. Evaluate boosted ranker on one held-out battery

def evaluate_xgb_ranker(
    model,
    test_pack,
    encoder,
):
    rows = []
    n_ops = test_pack["n_ops"]

    for t in range(n_ops - 1):
        candidate_indices = np.arange(
            t + 1,
            n_ops,
            dtype=np.int64,
        )

        Xq = pair_feature_matrix(
            test_pack,
            t,
            candidate_indices,
            encoder,
        )

        scores = model.predict(Xq)

        h1, h3, mrr = expected_tie_metrics(
            scores,
            target_position=0,
        )

        rows.append({
            "position": int(t),
            "candidate_size": int(
                len(candidate_indices)
            ),
            "hits1": float(h1),
            "hits3": float(h3),
            "mrr": float(mrr),
        })

    detail = pd.DataFrame(rows)

    metrics = {
        "hits1": float(detail["hits1"].mean()),
        "hits3": float(detail["hits3"].mean()),
        "mrr": float(detail["mrr"].mean()),
    }

    return metrics, detail


## 16. Interpretable training-only and topology heuristics

The class-transition prior uses only the nine training batteries.

The graph-distance and shared-component heuristics use the held-out battery's recorded fixation graph and operation-to-component references, but no learned parameters. They therefore provide direct tests of whether simple physical proximity or immediate component continuation predicts the recorded next operation.


In [ ]:
# 16. Heuristic definitions

def operation_transition_signature(op_id):
    r = ops_by_id.loc[int(op_id)]
    skill = str(r.skill_id)

    if pd.notna(r.fixation_id):
        fi = fix_info_raw[int(r.fixation_id)]
        return (
            "fix",
            skill,
            str(fi["class"]),
            str(part_class_raw[int(fi["fromPartID"])]),
            str(part_class_raw[int(fi["toPartID"])]),
        )

    return (
        "part",
        skill,
        str(part_class_raw[int(r.part_id)]),
    )


def fit_transition_prior(
    train_batteries,
    alpha=5.0,
):
    pair_counts = Counter()
    current_counts = Counter()
    candidate_counts = Counter()
    all_candidate_sigs = set()

    total_candidates = 0

    for b in train_batteries:
        seq = chains[b]
        sigs = [
            operation_transition_signature(op_id)
            for op_id in seq
        ]

        for sig in sigs:
            candidate_counts[sig] += 1
            all_candidate_sigs.add(sig)
            total_candidates += 1

        for a, c in zip(sigs[:-1], sigs[1:]):
            pair_counts[(a, c)] += 1
            current_counts[a] += 1

    K = max(1, len(all_candidate_sigs))

    def score(cur_sig, cand_sig):
        prior = (
            candidate_counts[cand_sig] + 1.0
        ) / (
            total_candidates + K
        )

        numer = (
            pair_counts[(cur_sig, cand_sig)]
            + alpha * prior
        )
        denom = (
            current_counts[cur_sig]
            + alpha
        )

        return float(
            np.log(
                max(numer / denom, 1e-15)
            )
        )

    return score


def touched_nodes(pack, op_index):
    row = pack["op_graph_nodes"][op_index]
    nodes = [
        int(x)
        for x in row
        if int(x) >= 0
    ]
    return tuple(sorted(set(nodes)))


def all_pairs_shortest_paths(pack):
    n_nodes = len(pack["node_part_class"])

    adj = [
        set()
        for _ in range(n_nodes)
    ]

    edge_index = pack["edge_index"]

    for u, v in edge_index.T:
        u = int(u)
        v = int(v)

        if u != v:
            adj[u].add(v)
            adj[v].add(u)

    dist = np.full(
        (n_nodes, n_nodes),
        np.inf,
        dtype=np.float32,
    )

    for s in range(n_nodes):
        dist[s, s] = 0.0
        q = deque([s])

        while q:
            u = q.popleft()
            du = dist[s, u]

            for v in adj[u]:
                if not np.isfinite(dist[s, v]):
                    dist[s, v] = du + 1.0
                    q.append(v)

    return dist


def graph_distance_score(
    current_nodes,
    candidate_nodes,
    dist,
):
    if len(current_nodes) == 0 or len(candidate_nodes) == 0:
        return -1e9

    d = min(
        dist[u, v]
        for u in current_nodes
        for v in candidate_nodes
    )

    if not np.isfinite(d):
        d = dist.shape[0] + 1.0

    # Smaller physical distance should rank higher.
    return float(-d)


def shared_component_score(
    current_nodes,
    candidate_nodes,
):
    return float(
        len(
            set(current_nodes)
            & set(candidate_nodes)
        )
    )


def random_expected_metrics(n):
    h1 = 1.0 / n
    h3 = min(3, n) / n
    harmonic = np.sum(
        1.0 / np.arange(
            1,
            n + 1,
            dtype=float,
        )
    )
    mrr = harmonic / n
    return float(h1), float(h3), float(mrr)


def evaluate_heuristics(
    test_battery,
    train_batteries,
    vocab,
):
    pack = build_battery_pack(
        test_battery,
        vocab,
        graph_mode="real",
        shuffle_id=0,
    )

    transition_score = fit_transition_prior(
        train_batteries
    )

    dist = all_pairs_shortest_paths(pack)

    seq = chains[test_battery]
    transition_sigs = [
        operation_transition_signature(op_id)
        for op_id in seq
    ]

    methods = {
        "ClassTransitionPrior": [],
        "GraphDistance": [],
        "SharedComponent": [],
        "UniformRandom": [],
    }

    for t in range(pack["n_ops"] - 1):
        candidate_indices = np.arange(
            t + 1,
            pack["n_ops"],
            dtype=np.int64,
        )

        cur_nodes = touched_nodes(pack, t)
        cur_sig = transition_sigs[t]

        transition_scores = np.asarray([
            transition_score(
                cur_sig,
                transition_sigs[j],
            )
            for j in candidate_indices
        ])

        graph_scores = np.asarray([
            graph_distance_score(
                cur_nodes,
                touched_nodes(pack, j),
                dist,
            )
            for j in candidate_indices
        ])

        overlap_scores = np.asarray([
            shared_component_score(
                cur_nodes,
                touched_nodes(pack, j),
            )
            for j in candidate_indices
        ])

        n = len(candidate_indices)

        for method, scores in [
            ("ClassTransitionPrior", transition_scores),
            ("GraphDistance", graph_scores),
            ("SharedComponent", overlap_scores),
        ]:
            h1, h3, mrr = expected_tie_metrics(
                scores,
                target_position=0,
            )

            methods[method].append({
                "position": int(t),
                "candidate_size": int(n),
                "hits1": float(h1),
                "hits3": float(h3),
                "mrr": float(mrr),
            })

        rh1, rh3, rmrr = random_expected_metrics(n)

        methods["UniformRandom"].append({
            "position": int(t),
            "candidate_size": int(n),
            "hits1": rh1,
            "hits3": rh3,
            "mrr": rmrr,
        })

    return {
        method: pd.DataFrame(rows)
        for method, rows in methods.items()
    }


## 17. Smoke test on one fold

This cell checks feature construction, one short boosted-ranker fit, and all three heuristics before the full run.


In [ ]:
# 17. E3 smoke test

TEST_BATTERY = sorted(chains)[0]
train_batteries = [
    b for b in sorted(chains)
    if b != TEST_BATTERY
]

vocab = build_fold_vocab(train_batteries)

X_train, y_train, qid_train, encoder = (
    build_xgb_training_data(
        train_batteries,
        vocab,
    )
)

print("Training rows:", X_train.shape[0])
print("Feature columns:", X_train.shape[1])
print("Ranking queries:", len(np.unique(qid_train)))

probe_params = dict(XGB_PARAMS)
probe_params["n_estimators"] = 5
probe_params["random_state"] = SEEDS[0]

probe = xgb.XGBRanker(
    **probe_params,
)
probe.fit(
    X_train,
    y_train,
    qid=qid_train,
    verbose=False,
)

test_pack = build_battery_pack(
    TEST_BATTERY,
    vocab,
    graph_mode="real",
    shuffle_id=0,
)

probe_metrics, _ = evaluate_xgb_ranker(
    probe,
    test_pack,
    encoder,
)

print("Probe XGB metrics:", probe_metrics)

heur = evaluate_heuristics(
    TEST_BATTERY,
    train_batteries,
    vocab,
)

for name, df in heur.items():
    print(
        name,
        {
            "hits1": float(df["hits1"].mean()),
            "hits3": float(df["hits3"].mean()),
            "mrr": float(df["mrr"].mean()),
        },
    )

del probe, X_train, y_train, qid_train
print("Smoke test passed.")


## 18. Full E3 LOPO experiment

The feature matrix is built once per LOPO fold and reused across the three XGB seeds. Heuristics are deterministic and are evaluated once per held-out product.

The partial CSVs make the run resumable.


In [ ]:
# 18. Full E3 experiment

OUT = Path("/content/battery_e3_strong_baselines_lopo")
OUT.mkdir(parents=True, exist_ok=True)

XGB_PARTIAL = OUT / "e3_xgb_seed_partial.csv"
HEUR_PARTIAL = OUT / "e3_heuristic_partial.csv"
XGB_DETAIL_PARTIAL = OUT / "e3_xgb_detail_partial.csv"
HEUR_DETAIL_PARTIAL = OUT / "e3_heuristic_detail_partial.csv"

if XGB_PARTIAL.exists():
    xgb_results = pd.read_csv(XGB_PARTIAL)
else:
    xgb_results = pd.DataFrame()

if HEUR_PARTIAL.exists():
    heuristic_results = pd.read_csv(HEUR_PARTIAL)
else:
    heuristic_results = pd.DataFrame()

if XGB_DETAIL_PARTIAL.exists():
    xgb_detail_all = pd.read_csv(XGB_DETAIL_PARTIAL)
else:
    xgb_detail_all = pd.DataFrame()

if HEUR_DETAIL_PARTIAL.exists():
    heuristic_detail_all = pd.read_csv(HEUR_DETAIL_PARTIAL)
else:
    heuristic_detail_all = pd.DataFrame()

battery_ids = sorted(chains)

def xgb_done(df, battery_id, seed):
    if len(df) == 0:
        return False
    return bool(
        (
            (df["battery_id"] == int(battery_id))
            & (df["seed"] == int(seed))
        ).any()
    )


def heuristics_done(df, battery_id):
    if len(df) == 0:
        return False
    expected = {
        "ClassTransitionPrior",
        "GraphDistance",
        "SharedComponent",
        "UniformRandom",
    }
    got = set(
        df.loc[
            df["battery_id"] == int(battery_id),
            "method",
        ].astype(str)
    )
    return expected.issubset(got)


start_time = time.time()

for fold_idx, test_battery in enumerate(
    battery_ids,
    start=1,
):
    train_batteries = [
        b for b in battery_ids
        if b != test_battery
    ]

    print(
        f"\nFold {fold_idx}/10 — held out: "
        f"{battery_names[test_battery]}"
    )

    vocab = build_fold_vocab(train_batteries)

    # ---------- deterministic heuristics ----------
    if not heuristics_done(
        heuristic_results,
        test_battery,
    ):
        heur = evaluate_heuristics(
            test_battery,
            train_batteries,
            vocab,
        )

        for method, df in heur.items():
            summary_row = {
                "battery_id": int(test_battery),
                "battery": battery_names[test_battery],
                "method": method,
                "hits1": float(df["hits1"].mean()),
                "hits3": float(df["hits3"].mean()),
                "mrr": float(df["mrr"].mean()),
            }

            heuristic_results = pd.concat(
                [
                    heuristic_results,
                    pd.DataFrame([summary_row]),
                ],
                ignore_index=True,
            )

            d = df.copy()
            d["battery_id"] = int(test_battery)
            d["battery"] = battery_names[test_battery]
            d["method"] = method

            heuristic_detail_all = pd.concat(
                [
                    heuristic_detail_all,
                    d,
                ],
                ignore_index=True,
            )

        heuristic_results.to_csv(
            HEUR_PARTIAL,
            index=False,
        )
        heuristic_detail_all.to_csv(
            HEUR_DETAIL_PARTIAL,
            index=False,
        )

        print("  heuristics done")

    # ---------- boosted ranker ----------
    missing_seeds = [
        seed
        for seed in SEEDS
        if not xgb_done(
            xgb_results,
            test_battery,
            seed,
        )
    ]

    if missing_seeds:
        print("  building full-candidate ranking matrix ...")

        X_train, y_train, qid_train, encoder = (
            build_xgb_training_data(
                train_batteries,
                vocab,
            )
        )

        test_pack = build_battery_pack(
            test_battery,
            vocab,
            graph_mode="real",
            shuffle_id=0,
        )

        print(
            "  X shape:",
            X_train.shape,
            "queries:",
            len(np.unique(qid_train)),
        )

        for seed in missing_seeds:
            model = build_xgb_model(seed)

            try:
                model.fit(
                    X_train,
                    y_train,
                    qid=qid_train,
                    verbose=False,
                )
            except Exception as exc:
                # Robust CPU fallback for environments whose XGBoost build
                # lacks CUDA ranking support.
                if "device" in model.get_params():
                    print(
                        "  GPU XGBoost fit failed; retrying on CPU:",
                        repr(exc),
                    )
                    cpu_params = dict(XGB_PARAMS)
                    cpu_params["device"] = "cpu"
                    cpu_params["tree_method"] = "hist"
                    cpu_params["random_state"] = int(seed)
                    model = xgb.XGBRanker(
                        **cpu_params,
                    )
                    model.fit(
                        X_train,
                        y_train,
                        qid=qid_train,
                        verbose=False,
                    )
                else:
                    raise

            metrics, detail = evaluate_xgb_ranker(
                model,
                test_pack,
                encoder,
            )

            row = {
                "battery_id": int(test_battery),
                "battery": battery_names[test_battery],
                "seed": int(seed),
                "hits1": metrics["hits1"],
                "hits3": metrics["hits3"],
                "mrr": metrics["mrr"],
                "training_rows": int(X_train.shape[0]),
                "feature_columns": int(X_train.shape[1]),
                "n_estimators": int(XGB_PARAMS["n_estimators"]),
            }

            xgb_results = pd.concat(
                [
                    xgb_results,
                    pd.DataFrame([row]),
                ],
                ignore_index=True,
            )

            detail = detail.copy()
            detail["battery_id"] = int(test_battery)
            detail["battery"] = battery_names[test_battery]
            detail["seed"] = int(seed)
            detail["method"] = "XGBRanker"

            xgb_detail_all = pd.concat(
                [
                    xgb_detail_all,
                    detail,
                ],
                ignore_index=True,
            )

            xgb_results.to_csv(
                XGB_PARTIAL,
                index=False,
            )
            xgb_detail_all.to_csv(
                XGB_DETAIL_PARTIAL,
                index=False,
            )

            print(
                f"  XGB seed={seed}: "
                f"H1={metrics['hits1']:.3f}, "
                f"H3={metrics['hits3']:.3f}, "
                f"MRR={metrics['mrr']:.3f}"
            )

            del model

        del X_train, y_train, qid_train, encoder, test_pack

elapsed = (time.time() - start_time) / 60.0

assert len(xgb_results) == 10 * len(SEEDS)
assert len(heuristic_results) == 10 * 4

print(
    f"\nCompleted E3 in {elapsed:.1f} minutes."
)


## 19. Product-level summaries, random-normalized skill score, and aggregate ranking

Neural/tree seeds are averaged within held-out product before comparison. Heuristics are already deterministic at product level.

The skill score is zero for uniform random ranking and one for perfect Hits@1.


In [ ]:
# 19. Product-level E3 summary

xgb_fold = (
    xgb_results
    .groupby(
        ["battery_id", "battery"],
        as_index=False,
    )
    .agg(
        hits1=("hits1", "mean"),
        hits3=("hits3", "mean"),
        mrr=("mrr", "mean"),
        hits1_seed_sd=("hits1", "std"),
    )
)

xgb_fold["method"] = "XGBRanker"

heur_fold = heuristic_results[
    [
        "battery_id",
        "battery",
        "method",
        "hits1",
        "hits3",
        "mrr",
    ]
].copy()

fold_all = pd.concat(
    [
        xgb_fold[
            [
                "battery_id",
                "battery",
                "method",
                "hits1",
                "hits3",
                "mrr",
            ]
        ],
        heur_fold,
    ],
    ignore_index=True,
)

random_h1 = (
    fold_all[
        fold_all["method"] == "UniformRandom"
    ][
        ["battery_id", "hits1"]
    ]
    .rename(
        columns={"hits1": "random_hits1"}
    )
)

fold_all = fold_all.merge(
    random_h1,
    on="battery_id",
    how="left",
)

fold_all["hits1_skill_score"] = (
    fold_all["hits1"]
    - fold_all["random_hits1"]
) / (
    1.0 - fold_all["random_hits1"]
)

aggregate = (
    fold_all
    .groupby("method", as_index=False)
    .agg(
        hits1=("hits1", "mean"),
        hits3=("hits3", "mean"),
        mrr=("mrr", "mean"),
        hits1_skill_score=("hits1_skill_score", "mean"),
    )
    .sort_values(
        "hits1",
        ascending=False,
    )
)

display(aggregate)
display(
    fold_all.sort_values(
        ["battery_id", "method"]
    )
)


## 20. Exact product-level comparisons among E3 baselines

Hits@1 is primary. The three planned comparisons test whether XGBRanker improves over each interpretable heuristic. Holm adjustment is applied across these three E3 Hits@1 contrasts. Hits@3 and MRR are supporting metrics.


In [ ]:
# 20. Exact paired E3 contrasts

def exact_two_sided_sign_flip(values):
    values = np.asarray(values, dtype=float)
    observed = abs(values.mean())
    total = 2 ** len(values)
    extreme = 0

    for signs in itertools.product(
        (-1.0, 1.0),
        repeat=len(values),
    ):
        stat = abs(
            np.mean(
                values
                * np.asarray(signs)
            )
        )
        if stat >= observed - 1e-15:
            extreme += 1

    return extreme / total


def bootstrap_mean_ci(
    values,
    n_boot=20000,
    seed=20270301,
):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)

    draws = rng.choice(
        values,
        size=(n_boot, len(values)),
        replace=True,
    ).mean(axis=1)

    return (
        float(np.percentile(draws, 2.5)),
        float(np.percentile(draws, 97.5)),
    )


def holm_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    m = len(pvalues)
    order = np.argsort(pvalues)

    adjusted_sorted = np.empty(
        m,
        dtype=float,
    )

    running = 0.0

    for rank, idx in enumerate(order):
        running = max(
            running,
            (m - rank) * pvalues[idx],
        )
        adjusted_sorted[rank] = min(
            1.0,
            running,
        )

    out = np.empty(
        m,
        dtype=float,
    )

    for rank, idx in enumerate(order):
        out[idx] = adjusted_sorted[rank]

    return out


wide = fold_all.pivot(
    index=[
        "battery_id",
        "battery",
    ],
    columns="method",
    values=[
        "hits1",
        "hits3",
        "mrr",
    ],
)

comparison_methods = [
    "ClassTransitionPrior",
    "GraphDistance",
    "SharedComponent",
]

rows = []

for metric_idx, metric in enumerate(
    ["hits1", "hits3", "mrr"]
):
    for j, method in enumerate(
        comparison_methods
    ):
        d = (
            wide[(metric, "XGBRanker")]
            - wide[(metric, method)]
        ).to_numpy(float)

        ci = bootstrap_mean_ci(
            d,
            seed=20270301
            + 100 * metric_idx
            + j,
        )

        rows.append({
            "metric": metric,
            "comparison": (
                f"XGBRanker - {method}"
            ),
            "mean_difference": float(
                d.mean()
            ),
            "ci_low": ci[0],
            "ci_high": ci[1],
            "positive_products": int(
                (d > 0).sum()
            ),
            "negative_products": int(
                (d < 0).sum()
            ),
            "zero_products": int(
                (d == 0).sum()
            ),
            "p_exact": float(
                exact_two_sided_sign_flip(d)
            ),
        })

contrast_stats = pd.DataFrame(rows)
contrast_stats["p_holm_hits1"] = np.nan

mask = (
    contrast_stats["metric"]
    == "hits1"
)

contrast_stats.loc[
    mask,
    "p_holm_hits1",
] = holm_adjust(
    contrast_stats.loc[
        mask,
        "p_exact",
    ].to_numpy(float)
)

display(contrast_stats)


## 21. Candidate-set-size stratification

This descriptive analysis asks whether a method's apparent strength is confined to easy states with few remaining operations.

For XGBRanker, the three seed-level state results are averaged first within `(product, position)`. Then method performance is averaged within product and candidate-size bin, followed by a product-balanced mean across products represented in that bin.


In [ ]:
# 21. Candidate-size stratification

def candidate_bin(n):
    for lo, hi, label in CANDIDATE_BINS:
        if lo <= n <= hi:
            return label
    raise ValueError(n)


xgb_state = (
    xgb_detail_all
    .groupby(
        [
            "battery_id",
            "battery",
            "position",
            "candidate_size",
        ],
        as_index=False,
    )
    .agg(
        hits1=("hits1", "mean"),
        hits3=("hits3", "mean"),
        mrr=("mrr", "mean"),
    )
)

xgb_state["method"] = "XGBRanker"

heur_state = heuristic_detail_all[
    [
        "battery_id",
        "battery",
        "position",
        "candidate_size",
        "hits1",
        "hits3",
        "mrr",
        "method",
    ]
].copy()

state_all = pd.concat(
    [
        xgb_state,
        heur_state,
    ],
    ignore_index=True,
)

state_all["candidate_bin"] = (
    state_all["candidate_size"]
    .map(candidate_bin)
)

product_bin = (
    state_all
    .groupby(
        [
            "method",
            "battery_id",
            "battery",
            "candidate_bin",
        ],
        as_index=False,
    )
    .agg(
        hits1=("hits1", "mean"),
        hits3=("hits3", "mean"),
        mrr=("mrr", "mean"),
        states=("position", "count"),
    )
)

bin_order = [
    x[2]
    for x in CANDIDATE_BINS
]

stratified = (
    product_bin
    .groupby(
        [
            "method",
            "candidate_bin",
        ],
        as_index=False,
    )
    .agg(
        hits1=("hits1", "mean"),
        hits3=("hits3", "mean"),
        mrr=("mrr", "mean"),
        products=("battery_id", "nunique"),
        states=("states", "sum"),
    )
)

stratified["candidate_bin"] = pd.Categorical(
    stratified["candidate_bin"],
    categories=bin_order,
    ordered=True,
)

stratified = stratified.sort_values(
    [
        "candidate_bin",
        "hits1",
    ],
    ascending=[
        True,
        False,
    ],
)

display(stratified)


## 22. Save E3 outputs

Upload `e3_strong_baselines_results.zip` back to ChatGPT. It contains the seed-level boosted-ranker results, deterministic heuristic results, product-level summaries, exact paired contrasts, candidate-size stratification, and metadata.


In [ ]:
# 22. Save all E3 outputs

xgb_results.to_csv(
    OUT / "e3_xgb_seed_results.csv",
    index=False,
)

heuristic_results.to_csv(
    OUT / "e3_heuristic_product_results.csv",
    index=False,
)

fold_all.to_csv(
    OUT / "e3_fold_all_methods.csv",
    index=False,
)

aggregate.to_csv(
    OUT / "e3_aggregate.csv",
    index=False,
)

contrast_stats.to_csv(
    OUT / "e3_contrasts.csv",
    index=False,
)

stratified.to_csv(
    OUT / "e3_candidate_size_stratified.csv",
    index=False,
)

product_bin.to_csv(
    OUT / "e3_candidate_size_product_bin.csv",
    index=False,
)

xgb_detail_all.to_csv(
    OUT / "e3_xgb_state_detail.csv",
    index=False,
)

heuristic_detail_all.to_csv(
    OUT / "e3_heuristic_state_detail.csv",
    index=False,
)

metadata = {
    "experiment": (
        "E3 strong non-graph ranking baseline "
        "and interpretable heuristics"
    ),
    "independent_unit": (
        "held-out battery product (n=10)"
    ),
    "xgb_seeds": [
        int(x)
        for x in SEEDS
    ],
    "xgb_params": {
        k: (
            int(v)
            if isinstance(v, np.integer)
            else float(v)
            if isinstance(v, np.floating)
            else v
        )
        for k, v in XGB_PARAMS.items()
    },
    "transition_prior_alpha": 5.0,
    "primary_endpoint": "Hits@1",
    "notes": (
        "XGBRanker uses all remaining candidates in every "
        "training ranking group and only non-adjacency operation "
        "features. GraphDistance and SharedComponent are "
        "deterministic held-out physical-graph heuristics. "
        "UniformRandom is analytical."
    ),
}

with open(
    OUT / "e3_metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
    )

LIGHT = Path(
    "/content/e3_strong_baselines_results_light"
)

if LIGHT.exists():
    shutil.rmtree(LIGHT)

LIGHT.mkdir(
    parents=True,
    exist_ok=True,
)

for fn in [
    "e3_xgb_seed_results.csv",
    "e3_heuristic_product_results.csv",
    "e3_fold_all_methods.csv",
    "e3_aggregate.csv",
    "e3_contrasts.csv",
    "e3_candidate_size_stratified.csv",
    "e3_candidate_size_product_bin.csv",
    "e3_metadata.json",
]:
    shutil.copy2(
        OUT / fn,
        LIGHT / fn,
    )

zip_path = shutil.make_archive(
    "/content/e3_strong_baselines_results",
    "zip",
    root_dir=LIGHT,
)

print("Saved:", zip_path)
for p in sorted(LIGHT.iterdir()):
    print(" -", p.name)
